# 📘 Project Starter: From DeepAR to Discrete Sequence Modeling

## Generalizing the original ETTh1 notebook

This project is a direct continuation of the original notebook on **probabilistic time-series forecasting with RNNs, LSTMs and Transformers (DeepAR-style)**.

In the original notebook, you worked with:

- the **ETTh1** dataset,
- a **univariate OT signal**,
- overlapping windows extracted from the full time series,
- and autoregressive models that predicted **Gaussian parameters** at each time step.

There, the model learned something like

$$x_t \mid x_{1:t-1} \sim \mathcal{N}(\mu_t, \sigma_t^2)$$

That is already a probabilistic model, and it is a very important one.  
But it also imposes a strong assumption: the conditional distribution of the next value is summarized by a **single mean** and a **single standard deviation**.

In this follow-up project, you will replace that Gaussian output with a **discrete probabilistic model**.

Instead of predicting:

- a mean,
- and a variance,

you will predict:

- a full probability vector over **128 bins**.

This leads to a much richer notion of uncertainty.

---

## Why this is interesting

A Gaussian output is convenient, but it is fundamentally limited:

- it is **unimodal**,
- it is **symmetric** around the mean,
- and it cannot easily represent more complex futures.

In contrast, a categorical distribution over bins can represent:

- **multiple peaks**,
- **skewed uncertainty**,
- **heavy tails**,
- and more irregular shapes.

So one of the key messages of this project is:

> **Discretization can improve uncertainty quantification because it removes the Gaussian shape constraint.**

---

## Connection to LLMs

This project also gives a very modern perspective on forecasting.

Large language models are trained to do:

> given previous tokens, predict the next token.

Mathematically:

$$P(w_t \mid w_1, \dots, w_{t-1})$$

After discretization, your forecasting model does exactly the same thing:

$$P(z_t \mid z_1, \dots, z_{t-1}),$$

where:

- \(z_t\) is no longer a word token,
- but the **index of the quantized signal bin** at time step \(t\).

So after quantization:

- a signal value becomes a **token**,
- the embedding layer becomes a **token embedding table**,
- forecasting becomes **next-token prediction**,
- and autoregressive sampling becomes **sequence generation**.

This is why the project is a beautiful bridge between:

- classical time-series forecasting,
- DeepAR-style probabilistic modeling,
- and the generative logic behind **LLMs**.

---

## What “next-token prediction” means in practice

Suppose an LLM sees:

> “The weather tomorrow will be ...”

It produces a probability distribution over the next token.  
Maybe it assigns high probability to words like:

- “sunny”
- “rainy”
- “cloudy”

Then a token is selected, and that token is fed back into the model to continue generation.

Your forecasting model will do the same thing, but with quantized signal values.

If the recent context suggests several plausible futures, the model may assign probability mass to several different bins.  
That is precisely why this approach can capture more complex uncertainty than a Gaussian model.

---

## Project objectives

You will:

1. reuse the **ETTh1 preprocessing pipeline** from the original notebook,
2. quantize the normalized signal into **128 uniform bins**,
3. replace scalar inputs by **learned embeddings**,
4. train autoregressive models with a **categorical next-token objective**,
5. generate forecasts by **sampling tokens autoregressively**,
6. compare the results with the original Gaussian models.

---

## Deliverables

You should report:

- how you discretized the signal,
- how you defined your embedding-based model,
- how training changed relative to the Gaussian version,
- how forecasts compare qualitatively and quantitatively,
- and whether the discrete formulation improves uncertainty modeling.


## 📡 Part I. ETTh1 dataset and preprocessing

To stay close to the original notebook, you should reuse the same general preprocessing logic:

1. Load `ETTh1.csv`
2. Extract the **OT** column
3. Normalize / standardize the signal
4. Optionally subsample it
5. Build overlapping windows for training and testing

The goal is that the new project remains directly comparable to the original Gaussian notebook.

### Important adaptation

In the original notebook, the model consumed a **continuous scalar** at each time step.

In this project, after normalization, you will convert each scalar value into a **token index** between 0 and 127.

So the pipeline becomes:

continuous series → normalized series → quantized bins → token sequence


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------
# 1. Load ETTh1 and extract the OT series
# --------------------------------------------------------
# Assumes ETTh1.csv is available in the working directory

df = pd.read_csv("ETTh1.csv")
series = df["OT"].values.astype(np.float32)

# --------------------------------------------------------
# 2. Normalize / standardize
# --------------------------------------------------------
# You may keep the same convention as in the original notebook.
# For example: standardization to zero mean and unit variance.

mean = series.mean()
std = series.std()
series = (series - mean) / std

# --------------------------------------------------------
# 3. Optional subsampling
# --------------------------------------------------------
subsample_factor = 10
series_sub = series[::subsample_factor]
T_total = len(series_sub)

print(f"Total length after subsampling: {T_total}")
print(f"First few values: {series_sub[:10]}")


Total length after subsampling: 1742
First few values: [2.0085132  0.77676684 0.5303475  0.8095683  0.85882866 1.285831
 1.5732229  0.7110473  0.9738088  1.8278133 ]


## 🔢 Part II. Uniform quantization into 128 bins

We now transform the normalized signal into **128 discrete levels**.

This is the first big conceptual step of the project.

### Why quantize?

Because once values are turned into bin indices, the forecasting task becomes a **token prediction problem**.

Instead of predicting a Gaussian distribution over real numbers, the model predicts a categorical distribution over:

$$\{0,1,2,\dots,127\}.$$

### Why can this improve uncertainty?

Because a categorical distribution over bins can represent much more than a Gaussian:

- not just one “central” future,
- but multiple plausible next values,
- with arbitrary asymmetric probabilities.

So discretization is not just a technical trick.  
It changes the **shape of the uncertainty model**.

### Instruction

For the main part of the project, use **uniform bins**.

Later, in the bonus section, you will test **quantile bins**.


In [3]:
NUM_BINS = 128

# --------------------------------------------------------
# Uniform quantization
# --------------------------------------------------------
# We quantize using the full normalized range observed in training.
# A good practice is to compute the quantization boundaries using
# the training split only.

# Example placeholder:
train_min = series_sub.min()
train_max = series_sub.max()

def quantize_uniform(x, x_min, x_max, num_bins=128):
    x_scaled = (x - x_min) / (x_max - x_min + 1e-8)
    bins = np.floor(num_bins * x_scaled).astype(int)
    bins = np.clip(bins, 0, num_bins - 1)
    return bins

def dequantize_uniform(bins, x_min, x_max, num_bins=128):
    # Map each bin to its center in the normalized space
    centers = (bins + 0.5) / num_bins
    return x_min + centers * (x_max - x_min)

# Example:
series_bins = quantize_uniform(series_sub, train_min, train_max, num_bins=NUM_BINS)

print("Unique bins used:", len(np.unique(series_bins)))
print("First 20 bin indices:", series_bins[:20])


Unique bins used: 121
First 20 bin indices: [ 89  62  56  63  64  73  80  60  66  85  89  85  85  89  89  79 106  86
  91  91]


## 📚 A short pause: why this now looks like an LLM problem

After quantization, your time series is a sequence such as:

`[41, 42, 42, 45, 47, 46, 44, ...]`

This is directly analogous to a text sequence such as:

`["the", "weather", "today", "is", "very", "warm"]`

In a language model:

- each token has an integer ID,
- that ID is mapped to a learned embedding vector,
- the model reads the sequence of embeddings,
- then predicts a distribution over the next token.

In this project:

- each quantized signal value has an integer ID,
- that ID is mapped to a learned embedding vector,
- the model reads the sequence of embeddings,
- then predicts a distribution over the next bin.

So when you train with cross-entropy on the next bin, you are doing **exactly the same learning principle** as next-token prediction in an LLM.

That does **not** mean the architectures are identical in every detail.  
But it does mean the **core probabilistic task** is the same.


## 🪟 Part III. Build overlapping windows

As in the original notebook, we will extract many windows from the full time series.

Each training example should look like:

- input:  $z_0, z_1, \dots, z_{T-1}$
- target: $z_1, z_2, \dots, z_T$

That is, each time step tries to predict the **next token**.

This is the same autoregressive shift used in language modeling.


In [4]:
# --------------------------------------------------------
# Build overlapping windows from the quantized series
# --------------------------------------------------------

T = 200      # sequence length, as in the original notebook
N = 5000     # number of windows to sample

if T_total <= T + 1:
    raise ValueError("Not enough points after subsampling to create windows.")

rng = np.random.default_rng(0)
starts = rng.integers(0, T_total - (T + 1), size=N)

X = np.stack([series_bins[s:s+T] for s in starts], axis=0)
Y = np.stack([series_bins[s+1:s+T+1] for s in starts], axis=0)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

# TODO:
# Split into train / test sets in a way consistent with the original notebook.


X shape: (5000, 200)
Y shape: (5000, 200)


## 🧩 Part IV. Dataset object

Notice the key difference relative to the Gaussian notebook:

- before: values were floating-point inputs,
- now: values are **integer token IDs**.

That means PyTorch tensors for the inputs should typically use `dtype=torch.long`,
because they are indexing rows of an embedding matrix.


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class TokenSequenceDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.Y = torch.tensor(Y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Example:
# train_dataset = TokenSequenceDataset(X_train, Y_train)
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


## 🧠 Part V. Embeddings: replacing scalar values by learned vectors

This is the second major conceptual step.

In the original notebook, the model received a scalar value \(x_t\) at each time step.

Now it will receive a **learned embedding vector** \(e_t\).

### Why embeddings?

A bin index such as `57` is just an integer label.  
By itself, it has no expressive representation.

The embedding matrix learns a vector for each bin:

$$57 \longrightarrow e_{57} \in \mathbb{R}^d $$

This is exactly what LLMs do with words or subword tokens.

### Why is this useful?

Because the model can learn relationships between bins in representation space.  
For example, nearby or frequently co-occurring bins may develop related embeddings.

So the embedding layer is not just an implementation detail.  
It is what allows the model to build an internal, learned geometry of quantized values.


In [6]:
import torch.nn as nn
import torch.nn.functional as F

class DiscreteRNN(nn.Module):
    def __init__(self, num_bins=128, d_model=64, hidden_dim=32, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(num_bins, d_model)
        self.rnn = nn.RNN(
            input_size=d_model,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            nonlinearity="relu",
            batch_first=True
        )
        self.output = nn.Linear(hidden_dim, num_bins)

    def forward(self, x):
        # x: (batch, seq_len) of integer token IDs
        emb = self.embedding(x)          # (batch, seq_len, d_model)
        h, hidden = self.rnn(emb)        # (batch, seq_len, hidden_dim)
        logits = self.output(h)          # (batch, seq_len, num_bins)
        return logits, hidden


## ✍️ Your task: adapt the other architectures too

To remain aligned with the original notebook, do **not** stop at the RNN.

You should adapt at least one additional architecture from the original notebook, ideally:

- **LSTM**
- and/or the **hybrid LSTM–Transformer** model

The modification principle is always the same:

1. replace continuous scalar input by token IDs,
2. add an embedding layer,
3. output logits over 128 bins,
4. train with cross-entropy instead of Gaussian loss.

This mirrors the structure of the original notebook while changing the probabilistic output space.


In [7]:
class DiscreteLSTM(nn.Module):
    def __init__(self, num_bins=128, d_model=64, hidden_dim=32, n_layers=1, drop_prob=0.3):
        super().__init__()
        self.embedding = nn.Embedding(num_bins, d_model)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            dropout=drop_prob if n_layers > 1 else 0.0,
            batch_first=True
        )
        self.output = nn.Linear(hidden_dim, num_bins)

    def forward(self, x):
        emb = self.embedding(x)
        h, hidden = self.lstm(emb)
        logits = self.output(h)
        return logits, hidden

# TODO:
# Build a discrete version of the Transformer / LSTM-Transformer section
# from the original notebook. The main design idea is:
# token IDs -> embedding -> sequence model -> logits over bins


## 📉 Part VI. Training objective

In the original Gaussian notebook, the loss was based on predicting a continuous distribution.

Here, once the signal has been quantized, the correct probabilistic training objective is:

- **cross-entropy loss** on the next token.

That is, if the model outputs logits $\ell_t$ over 128 bins, we train it to maximize the probability assigned to the true next bin.

Mathematically, this is a categorical likelihood model.

This is again exactly the same principle used in LLMs.


In [8]:
def token_cross_entropy_loss(logits, targets):
    # logits:  (B, T, C)
    # targets: (B, T)
    B, T, C = logits.shape
    logits = logits.reshape(B * T, C)
    targets = targets.reshape(B * T)
    return F.cross_entropy(logits, targets)


## 🔁 Part VII. Training loop

You may reuse the organizational style of the original notebook, with “extended” training wrappers if you want.

The main change is simply the loss function and the input type.


In [9]:
class DiscreteRNNExtended(DiscreteRNN):
    def __init__(self, num_data_train, num_iter, sequence_length,
                 num_bins=128, d_model=64, hidden_dim=32, n_layers=1, lr=1e-3):
        super().__init__(num_bins=num_bins, d_model=d_model, hidden_dim=hidden_dim, n_layers=n_layers)
        self.num_train = num_data_train
        self.num_iter = num_iter
        self.sequence_length = sequence_length
        self.optim = torch.optim.Adam(self.parameters(), lr=lr)
        self.loss_during_training = []

    def trainloop(self, train_loader, device="cpu"):
        # TO DO: implement the training loop, similar to the original notebook.
        # The main difference is that the loss is now token_cross_entropy_loss
        # instead of MSE, and the model outputs logits over bins instead of continuous values.
        


SyntaxError: incomplete input (1468378413.py, line 15)

## 🎲 Part VIII. Autoregressive sampling and forecasting

This is where the LLM analogy becomes especially vivid.

In text generation:

1. the model predicts a distribution over the next token,
2. one token is selected or sampled,
3. that token is appended to the sequence,
4. the process repeats.

In this forecasting project, you will do the same thing with signal tokens.

### Important conceptual point

A Gaussian model samples a real value from $\mathcal{N}(\mu_t, \sigma_t^2)$.

A discrete model samples a token from:

$$ \text{Categorical}(p_1, \dots, p_{128}) $$

This can produce much richer forecast distributions, especially when the future is ambiguous.


In [ ]:
@torch.no_grad()
def sample_tokens(model, seed_seq, steps=50, temperature=1.0, device="cpu"):
    model.eval()
    model.to(device)

    seq = seed_seq.clone().to(device)  # shape: (T0,)

    for _ in range(steps):
        x = seq.unsqueeze(0)                 # (1, T)
        logits, _ = model(x)                 # (1, T, C)
        next_logits = logits[0, -1] / temperature
        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # (1,)
        seq = torch.cat([seq, next_token], dim=0)

    return seq.cpu()

# Example usage:
# seed = torch.tensor(X_test[0, :50], dtype=torch.long)
# pred_tokens = sample_tokens(model, seed, steps=100)


## 🔄 Convert tokens back to real values

To visualize forecasts, you should map sampled bins back to approximate continuous values using the bin centers.

This is similar to “decoding” tokens back into a signal representation.


In [ ]:
def tokens_to_values(token_seq, x_min, x_max, num_bins=128):
    token_seq = np.asarray(token_seq)
    return dequantize_uniform(token_seq, x_min, x_max, num_bins=num_bins)


## 📊 Part IX. Evaluation

To stay close to the original notebook, you should compare models using familiar evaluation blocks.

For example:

### 1. One-step-ahead evaluation
Use teacher forcing:
- provide the true previous tokens,
- predict the next-token distribution,
- compute the loss and possibly the expected-value forecast.

### 2. Autoregressive forecasting beyond the context window
As in the original notebook:
- give the first \(T_{test}\) ground-truth tokens,
- then roll forward using the model’s own sampled predictions.

### 3. Comparison with the Gaussian notebook
You may report:
- one-step MSE after decoding tokens to values,
- forecasting MSE,
- qualitative plots,
- and comments about uncertainty realism.

### Important reflection question
Even when pointwise MSE is similar, does the discrete model produce **more expressive uncertainty**?


In [ ]:
# TODO:
# Recreate the original notebook's evaluation sections for the discrete model:
# - one-step-ahead evaluation
# - autoregressive long-horizon forecasting
# - final comparison table across models


## 🌟 Bonus section

### Bonus 1 — Quantile binning

In the main project you must start with **uniform bins**.

Now test a different idea:

- define bins using **quantiles** of the training data,
- so that each bin contains roughly the same number of samples.

This often gives:
- better resolution in dense regions,
- but less interpretability in terms of equal-width values.

Question:
> Does quantile binning improve forecasting or uncertainty quality?

---

### Bonus 2 — Number of bins

Try:
- 64
- 128
- 256

What changes in:
- training stability,
- forecast sharpness,
- uncertainty richness?

---

### Bonus 3 — Temperature sampling

As in language models, you can control diversity at generation time:

- lower temperature → sharper, more deterministic predictions
- higher temperature → more diverse, more exploratory predictions

---

### Bonus 4 — Expected-value decoding

Instead of sampling one token, compute the expected value under the predicted categorical distribution.

This gives a smoother forecast and may improve pointwise error, but may reduce diversity.


## ✅ Final interpretation

By the end of this project, you should understand that:

- forecasting can be framed as **sequence generation**,
- quantization converts continuous signals into **tokens**,
- embeddings give those tokens learned representations,
- cross-entropy training corresponds to **categorical next-token prediction**,
- and discretization can improve uncertainty modeling because it allows **more complex conditional distributions** than a Gaussian.

In other words:

> you are taking ideas from probabilistic forecasting and reinterpreting them through the lens of modern sequence modeling and LLMs.
